# JupyterLite で学ぶ pingouin 統計検定 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
統計検定ライブラリ **pingouin** を使って、t 検定・分散分析・相関・カイ二乗検定・回帰分析を
「1 行で・効果量つきで」実行する方法を学ぶためのチュートリアルです。

## 対象者
- 平均・分散・検定の考え方（帰無仮説と p 値）を学び始めた方
- `scipy.stats` の検定結果（統計量と p 値だけ）に物足りなさを感じている方
- 経済データ（賃金・地域・教育など）で統計検定を実践したい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. pingouin とは（scipy.stats・statsmodels との違い）
2. 分析用データの準備（合成した賃金データ）
3. 記述統計と検定の前提チェック（正規性・等分散性）
4. t 検定と効果量・検出力
5. ノンパラメトリック検定
6. 分散分析（ANOVA）と多重比較
7. 相関分析
8. カイ二乗検定
9. 回帰分析（線形・ロジスティック）
10. ベイズファクター
11. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。途中を飛ばすと変数が未定義になります。
- 各章の最後に **練習問題** があります。「解答欄」に自分で書いてから「解答例」を開いて確認しましょう。
- 統計の結果は **DataFrame** として返ってくるので、pandas の操作（列の選択・保存）がそのまま使えます。

---
## 0. 環境準備（JupyterLite 用）

pingouin は Pyodide に同梱されていないため、`piplite` で PyPI からインストールします（純 Python なので数秒で入ります）。
日本語のグラフを描くために `japanize-matplotlib-jlite` も入れておきます。

In [ ]:
# JupyterLite 用のパッケージインストール（初回は数十秒かかることがあります）
try:
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "scipy", "statsmodels", "pingouin", "japanize-matplotlib-jlite"])
    print("piplite でのインストールが完了しました")
except ImportError:
    print("piplite がない環境（ローカルの Jupyter）なのでスキップしました")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語フォント（pyplot の後に import）
import pingouin as pg
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
print("pingouin バージョン:", pg.__version__)
print("pandas バージョン  :", pd.__version__)

---
## 1. pingouin とは

**pingouin** は、`scipy.stats` と `statsmodels` の上に作られた「統計検定を簡潔に書くための」ライブラリです。

| 項目 | `scipy.stats` | **pingouin** |
|---|---|---|
| 結果の形式 | 統計量と p 値のタプル | **DataFrame**（列名つきの表） |
| 効果量（Cohen's d など） | 自分で計算 | **自動で付く** |
| 信頼区間 | 自分で計算 | **自動で付く** |
| 検出力（power） | `statsmodels` が必要 | **自動で付く** |
| ベイズファクター | なし | **自動で付く** |
| 多重比較（Tukey など） | `statsmodels` が必要 | 1 行で実行 |

同じ t 検定を両方で書いて比べてみましょう。

In [ ]:
rng = np.random.default_rng(0)
a = rng.normal(50, 10, 30)   # グループ A の 30 人分のデータ
b = rng.normal(55, 10, 30)   # グループ B の 30 人分のデータ

# scipy.stats：統計量と p 値だけが返る
t_stat, p_value = stats.ttest_ind(a, b)
print(f"scipy.stats : t = {t_stat:.3f}, p = {p_value:.4f}")

# pingouin：自由度・信頼区間・効果量・検出力・ベイズファクターまで 1 行で
result = pg.ttest(a, b)
print(result.round(3))

`pg.ttest()` が返す表の列は次のとおりです。

| 列 | 意味 |
|---|---|
| `T` | t 統計量 |
| `dof` | 自由度（Welch の補正が入ると小数になる） |
| `alternative` | 対立仮説（`two-sided` = 両側検定） |
| `p_val` | p 値 |
| `CI95` | 平均の差の 95% 信頼区間 |
| `cohen_d` | 効果量 Cohen's d（差の大きさを標準偏差単位で表す） |
| `power` | 検出力（この標本サイズで真の差を検出できる確率） |
| `BF10` | ベイズファクター（対立仮説の証拠の強さ） |

結果は DataFrame なので、必要な値だけ取り出すこともできます。

In [ ]:
print("p 値      :", round(result["p_val"].iloc[0], 4))
print("Cohen's d :", round(result["cohen_d"].iloc[0], 3))
print("検出力    :", round(result["power"].iloc[0], 3))

---
## 2. 分析用データの準備（合成した賃金データ）

このチュートリアルでは、乱数で作った **架空の賃金データ**（400 人分）を使います。
実際のデータではありませんが、「性別・地域・教育年数・勤続年数・雇用形態が年収に影響する」という
経済学でよく扱う構造を組み込んであります。`rng = np.random.default_rng(42)` で乱数を固定しているので、
誰が実行しても同じデータになります。

| 列 | 意味 |
|---|---|
| `gender` | 性別（男性 / 女性） |
| `region` | 地域（関東 / 中部 / 九州） |
| `education` | 教育年数（12 = 高卒、16 = 大卒 など） |
| `experience` | 勤続年数 |
| `emp_type` | 雇用形態（正規 / 非正規） |
| `wage` | 年収（万円） |

In [ ]:
rng = np.random.default_rng(42)
n = 400

gender = rng.choice(["男性", "女性"], size=n)
region = rng.choice(["関東", "中部", "九州"], size=n, p=[0.5, 0.3, 0.2])
education = rng.choice([12, 14, 16, 18], size=n, p=[0.3, 0.2, 0.4, 0.1])
experience = rng.integers(0, 35, size=n)
is_regular = rng.random(n) < np.where(education >= 16, 0.8, 0.55)

wage = (
    150
    + 18 * education
    + 9 * experience - 0.12 * experience ** 2
    + np.where(gender == "女性", -45, 0)
    + np.select([region == "関東", region == "中部"], [40, 10], default=0)
    + np.where(is_regular, 60, 0)
    + rng.normal(0, 55, size=n)
)

df = pd.DataFrame({
    "gender": gender,
    "region": region,
    "education": education,
    "experience": experience,
    "emp_type": np.where(is_regular, "正規", "非正規"),
    "wage": wage.round(1),
})
print(df.head())
print("\nデータの大きさ:", df.shape)

In [ ]:
# グループごとの平均年収を眺めておく
print(df.groupby("gender")["wage"].agg(["count", "mean", "std"]).round(1))
print()
print(df.groupby("region")["wage"].agg(["count", "mean", "std"]).round(1))
print()
print(df.groupby("emp_type")["wage"].agg(["count", "mean", "std"]).round(1))

---
## 3. 記述統計と検定の前提チェック

t 検定や分散分析は「データが正規分布に近い」「グループ間で分散が等しい」ことを前提にしています。
pingouin には前提をチェックする関数が用意されています。

| 関数 | 検定 | 帰無仮説 |
|---|---|---|
| `pg.normality()` | Shapiro-Wilk 検定 | データは正規分布に従う |
| `pg.homoscedasticity()` | Levene 検定 | グループ間の分散は等しい |

`normal` / `equal_var` 列が `True` なら前提が満たされている（棄却されない）と判断します。

In [ ]:
# 性別ごとの正規性検定（Shapiro-Wilk）
print(pg.normality(data=df, dv="wage", group="gender").round(3))
print()
# 性別間の等分散性検定（Levene）
print(pg.homoscedasticity(data=df, dv="wage", group="gender").round(3))

In [ ]:
# ヒストグラムで分布の形を確認する
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, group) in zip(axes, df.groupby("gender")):
    ax.hist(group["wage"], bins=20, color="steelblue", edgecolor="white")
    ax.set_title(f"{name}の年収分布（n={len(group)}）")
    ax.set_xlabel("年収（万円）")
    ax.set_ylabel("人数")
plt.tight_layout()
plt.show()

In [ ]:
# Q-Q プロット：点が直線上に並べば正規分布に近い
ax = pg.qqplot(df["wage"], dist="norm")
ax.set_title("年収の Q-Q プロット")
plt.show()

標本サイズが大きい（数百以上）ときは、Shapiro-Wilk 検定はわずかなずれでも「正規ではない」と判定しがちです。
実務では **ヒストグラムや Q-Q プロットを見て判断する** ことと、
正規性が疑わしいときは **ノンパラメトリック検定（第 5 章）** も併用することをおすすめします。

### 練習問題 1

1. `experience`（勤続年数）について、地域（`region`）ごとに正規性検定を行ってください。
2. 年収（`wage`）について、雇用形態（`emp_type`）間で等分散性検定（Levene）を行ってください。
3. 雇用形態ごとの年収のヒストグラムを 1 枚の図に重ねて描いてください（`alpha=0.5` で半透明に）。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
print(pg.normality(data=df, dv="experience", group="region").round(3))

# 2
print(pg.homoscedasticity(data=df, dv="wage", group="emp_type").round(3))

# 3
fig, ax = plt.subplots(figsize=(7, 4))
for name, group in df.groupby("emp_type"):
    ax.hist(group["wage"], bins=20, alpha=0.5, label=name)
ax.set_xlabel("年収（万円）")
ax.set_ylabel("人数")
ax.legend()
plt.show()
```

</details>

---
## 4. t 検定と効果量・検出力

### 4.1 1 標本 t 検定：平均は特定の値と等しいか

「全体の平均年収は 450 万円と言えるか」を検定します。第 2 引数に比較したい値を渡します。

In [ ]:
print("標本平均:", round(df["wage"].mean(), 1), "万円")
print(pg.ttest(df["wage"], 450).round(3))

### 4.2 対応のない 2 標本 t 検定：2 つのグループの平均は等しいか

男性と女性の平均年収に差があるかを検定します。
pingouin は既定で **Welch の t 検定**（等分散を仮定しない版）を自動選択します（`correction="auto"`）。
Student の t 検定（等分散を仮定）にしたいときは `correction=False` を指定します。

In [ ]:
male = df.loc[df["gender"] == "男性", "wage"]
female = df.loc[df["gender"] == "女性", "wage"]

print("平均の差（男性 − 女性）:", round(male.mean() - female.mean(), 1), "万円\n")
print("Welch の t 検定（既定）:")
print(pg.ttest(male, female).round(3))
print("\nStudent の t 検定（等分散を仮定）:")
print(pg.ttest(male, female, correction=False).round(3))

`p_val` が 0.05 より小さければ「平均に差がない」という帰無仮説を棄却します。
`CI95` は平均の差の 95% 信頼区間で、0 を含まなければ有意差ありと同じ結論になります。
片側検定にしたいときは `alternative="greater"` / `"less"` を指定します。

In [ ]:
# 片側検定：男性の平均は女性より「大きい」か
print(pg.ttest(male, female, alternative="greater").round(3))

### 4.3 対応のある t 検定：同じ人の前後比較

同じ 40 人に研修を受けてもらい、研修前後のテスト得点を比べます。
同じ人の 2 回の測定なので **対応のある t 検定**（`paired=True`）を使います。

In [ ]:
rng2 = np.random.default_rng(7)
before = rng2.normal(60, 8, 40)
after = before + rng2.normal(3, 5, 40)   # 平均 3 点上がるように作ったデータ

training = pd.DataFrame({"before": before.round(1), "after": after.round(1)})
print(training.head())
print("\n平均の変化:", round((training["after"] - training["before"]).mean(), 2), "点\n")
print(pg.ttest(training["after"], training["before"], paired=True).round(3))

### 4.4 効果量（effect size）

p 値は「差があるかどうか」しか教えてくれません。標本が大きければ、ごく小さな差でも有意になります。
**効果量** は差の「大きさ」を標準偏差の単位で表す指標で、p 値と一緒に必ず報告しましょう。

| Cohen's d | 目安 |
|---|---|
| 0.2 | 小さい |
| 0.5 | 中くらい |
| 0.8 | 大きい |

`pg.compute_effsize()` でいろいろな効果量を計算できます。

In [ ]:
print("Cohen's d :", round(pg.compute_effsize(male, female, eftype="cohen"), 3))
print("Hedges' g :", round(pg.compute_effsize(male, female, eftype="hedges"), 3))
print("CLES      :", round(pg.compute_effsize(male, female, eftype="CLES"), 3),
      "← ランダムに選んだ男性の年収が女性より高い確率")

### 4.5 検出力（power）と必要な標本サイズ

**検出力** は「本当に差があるときに、その差を検出（有意と判定）できる確率」です。慣習的に 0.8 以上が望ましいとされます。
`pg.power_ttest()` は、効果量 `d`・標本サイズ `n`・有意水準 `alpha`・検出力 `power` のうち
**3 つを与えると残りの 1 つを計算** してくれます。

In [ ]:
# 効果量 0.5 を 1 群 40 人で検出できる確率
print("検出力:", round(pg.power_ttest(d=0.5, n=40, alpha=0.05), 3))

# 効果量 0.3 を検出力 0.8 で検出するには 1 群あたり何人必要か
print("必要な標本サイズ:", round(pg.power_ttest(d=0.3, power=0.8, alpha=0.05)), "人")

In [ ]:
# 検出力曲線：効果量ごとに、標本サイズと検出力の関係を描く
sizes = np.arange(5, 201, 5)
plt.figure(figsize=(7, 4))
for d in [0.2, 0.5, 0.8]:
    powers = [pg.power_ttest(d=d, n=n_, alpha=0.05) for n_ in sizes]
    plt.plot(sizes, powers, marker="o", markersize=3, label=f"d = {d}")
plt.axhline(0.8, color="red", linestyle="--", label="検出力 0.8")
plt.xlabel("1 群あたりの標本サイズ")
plt.ylabel("検出力")
plt.title("検出力曲線（有意水準 5%）")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 練習問題 2

1. 正規雇用と非正規雇用の平均年収に差があるか、Welch の t 検定で調べ、Cohen's d を報告してください。
2. 教育年数が 16 年以上（大卒以上）とそれ未満で、平均年収に差があるか検定してください。
3. 効果量 0.4 を検出力 0.9、有意水準 5% で検出するのに必要な 1 群あたりの標本サイズを求めてください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
regular = df.loc[df["emp_type"] == "正規", "wage"]
irregular = df.loc[df["emp_type"] == "非正規", "wage"]
res = pg.ttest(regular, irregular)
print(res.round(3))
print("Cohen's d:", round(res["cohen_d"].iloc[0], 3))

# 2
univ = df.loc[df["education"] >= 16, "wage"]
non_univ = df.loc[df["education"] < 16, "wage"]
print(pg.ttest(univ, non_univ).round(3))

# 3
print(round(pg.power_ttest(d=0.4, power=0.9, alpha=0.05)), "人")
```

</details>

---
## 5. ノンパラメトリック検定

正規分布を仮定できないとき（分布が大きく歪んでいる、外れ値が多い、順位データなど）は、
**順位** に基づくノンパラメトリック検定を使います。

| 状況 | パラメトリック | ノンパラメトリック |
|---|---|---|
| 対応のない 2 群 | t 検定 | **Mann-Whitney の U 検定** `pg.mwu()` |
| 対応のある 2 群 | 対応のある t 検定 | **Wilcoxon の符号順位検定** `pg.wilcoxon()` |
| 3 群以上 | 一元配置分散分析 | **Kruskal-Wallis 検定** `pg.kruskal()` |

In [ ]:
# Mann-Whitney の U 検定（男性 vs 女性）
print(pg.mwu(male, female).round(3))
print()
# Wilcoxon の符号順位検定（研修前後）
print(pg.wilcoxon(training["after"], training["before"]).round(3))
print()
# Kruskal-Wallis 検定（3 地域）
print(pg.kruskal(data=df, dv="wage", between="region").round(3))

`RBC`（rank-biserial correlation）と `CLES`（common language effect size）はノンパラメトリック検定用の効果量です。
p 値の解釈は t 検定と同じですが、帰無仮説は「2 群の **分布** が同じ」である点に注意してください。

---
## 6. 分散分析（ANOVA）と多重比較

### 6.1 一元配置分散分析

3 つ以上のグループの平均を一度に比較するには **分散分析** を使います。
「地域によって平均年収が異なるか」を調べてみましょう。`detailed=True` で平方和（SS）なども表示されます。

In [ ]:
print(pg.anova(data=df, dv="wage", between="region", detailed=True).round(3))

In [ ]:
# 地域ごとの年収の箱ひげ図
groups = [g["wage"].values for _, g in df.groupby("region")]
labels = [name for name, _ in df.groupby("region")]
plt.figure(figsize=(7, 4))
plt.boxplot(groups, tick_labels=labels)
plt.ylabel("年収（万円）")
plt.title("地域別の年収")
plt.show()

`np2`（偏イータ二乗）は分散分析の効果量で、「グループの違いで説明できる分散の割合」です（0.01: 小、0.06: 中、0.14: 大が目安）。

等分散性が疑わしいときは **Welch の分散分析** `pg.welch_anova()` を使います。

In [ ]:
print(pg.welch_anova(data=df, dv="wage", between="region").round(3))

### 6.2 多重比較（どのグループ間に差があるのか）

分散分析が有意でも「どこかに差がある」ことしか分かりません。どのペアに差があるかを調べるのが **多重比較** です。
ペアごとに t 検定を繰り返すと偽陽性が増えるので、p 値を補正します。

- `pg.pairwise_tests(..., padjust="bonf")`：ペアごとの t 検定 + Bonferroni 補正（`p_corr` 列が補正後の p 値）
- `pg.pairwise_tukey()`：Tukey の HSD 検定（分散分析の後の多重比較の定番）

In [ ]:
print("ペアごとの t 検定（Bonferroni 補正）:")
print(pg.pairwise_tests(data=df, dv="wage", between="region", padjust="bonf").round(3))
print("\nTukey の HSD 検定:")
print(pg.pairwise_tukey(data=df, dv="wage", between="region").round(3))

### 6.3 二元配置分散分析（要因が 2 つ）

`between` にリストを渡すと、2 つの要因（地域と性別）の **主効果** と **交互作用**（`region * gender`）を同時に検定できます。

In [ ]:
print(pg.anova(data=df, dv="wage", between=["region", "gender"]).round(3))

In [ ]:
# 交互作用プロット：地域ごとの平均年収を性別で分けて描く
means = df.groupby(["region", "gender"])["wage"].mean().unstack()
print(means.round(1))
means.plot(kind="bar", figsize=(7, 4))
plt.ylabel("平均年収（万円）")
plt.title("地域 × 性別の平均年収")
plt.xticks(rotation=0)
plt.legend(title="性別")
plt.show()

### 6.4 反復測定分散分析（同じ人を何度も測る）

同じ 30 人の従業員に、入社時・1 年後・2 年後の 3 時点で仕事満足度（0〜100）を尋ねたとします。
同じ人の繰り返し測定なので **反復測定分散分析** `pg.rm_anova()` を使います。データは「1 行 = 1 人 × 1 時点」の縦長（long）形式にします。

In [ ]:
rng3 = np.random.default_rng(3)
n_people = 30
base = rng3.normal(60, 10, n_people)
scores_3times = np.column_stack([
    base,                                        # 入社時
    base + 4 + rng3.normal(0, 5, n_people),      # 1 年後（平均 +4 点）
    base + 7 + rng3.normal(0, 5, n_people),      # 2 年後（平均 +7 点）
])
satisfaction = pd.DataFrame({
    "id": np.repeat(np.arange(n_people), 3),
    "time": np.tile(["入社時", "1年後", "2年後"], n_people),
    "score": scores_3times.ravel().round(1),     # 1 人の 3 時点が連続して並ぶ
})
print(satisfaction.head(6))
print()
print(pg.rm_anova(data=satisfaction, dv="score", within="time", subject="id").round(3))

In [ ]:
# 事後検定（対応ありのペア比較、Bonferroni 補正）
print(pg.pairwise_tests(data=satisfaction, dv="score", within="time", subject="id", padjust="bonf").round(3))

### 練習問題 3

1. 教育年数（`education`：12, 14, 16, 18 の 4 群）によって平均年収が異なるか、一元配置分散分析で検定してください。
2. 1 の結果について Tukey の HSD 検定で多重比較を行い、有意差のあるペア（`p_tukey < 0.05`）だけを表示してください。
3. 教育年数と雇用形態（`emp_type`）を要因とする二元配置分散分析を行い、交互作用が有意かどうか答えてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
print(pg.anova(data=df, dv="wage", between="education").round(3))

# 2
tukey = pg.pairwise_tukey(data=df, dv="wage", between="education")
print(tukey[tukey["p_tukey"] < 0.05].round(3))

# 3
res = pg.anova(data=df, dv="wage", between=["education", "emp_type"])
print(res.round(3))
print("交互作用の p 値:", round(res.loc[res["Source"] == "education * emp_type", "p_unc"].iloc[0], 3))
```

</details>

---
## 7. 相関分析

### 7.1 2 変数の相関

`pg.corr()` は相関係数に加えて、信頼区間・p 値・ベイズファクター・検出力を返します。
`method` で Pearson（既定）、Spearman、Kendall などを選べます。

In [ ]:
print("Pearson の相関（勤続年数と年収）:")
print(pg.corr(df["experience"], df["wage"]).round(3))
print("\nSpearman の順位相関:")
print(pg.corr(df["experience"], df["wage"], method="spearman").round(3))

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["experience"], df["wage"], alpha=0.5)
plt.xlabel("勤続年数")
plt.ylabel("年収（万円）")
plt.title("勤続年数と年収の関係")
plt.grid(alpha=0.3)
plt.show()

勤続年数と年収は「逆 U 字」の関係（年数とともに上がるが、あるところから頭打ち）になるように作ってあります。
直線の相関（Pearson）だけでは、こうした非線形の関係を見落とすことがあるので、必ず散布図も描きましょう。

### 7.2 相関行列（pairwise_corr）と偏相関

- `pg.pairwise_corr()`：複数の変数のすべてのペアについて相関を計算
- `pg.partial_corr()`：第 3 の変数（`covar`）の影響を取り除いた **偏相関**

In [ ]:
print(pg.pairwise_corr(df[["education", "experience", "wage"]]).round(3))

In [ ]:
# 教育年数の影響を取り除いたうえでの、勤続年数と年収の偏相関
print(pg.partial_corr(data=df, x="experience", y="wage", covar="education").round(3))

### 練習問題 4

1. 教育年数（`education`）と年収（`wage`）の Pearson 相関と Spearman 相関を求めてください。
2. 勤続年数（`experience`）の影響を取り除いた、教育年数と年収の偏相関を求めてください。
3. 教育年数と年収の散布図を、性別で色分けして描いてください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
print(pg.corr(df["education"], df["wage"]).round(3))
print(pg.corr(df["education"], df["wage"], method="spearman").round(3))

# 2
print(pg.partial_corr(data=df, x="education", y="wage", covar="experience").round(3))

# 3
plt.figure(figsize=(7, 4))
for name, group in df.groupby("gender"):
    plt.scatter(group["education"], group["wage"], alpha=0.5, label=name)
plt.xlabel("教育年数")
plt.ylabel("年収（万円）")
plt.legend()
plt.show()
```

</details>

---
## 8. カイ二乗検定（カテゴリ変数どうしの関係）

「地域と雇用形態は独立か（関係がないか）」のように、**2 つのカテゴリ変数の関係** を調べるには
**カイ二乗独立性検定** を使います。`pg.chi2_independence()` は 3 つの表を返します。

1. 期待度数（独立ならこうなるはずの度数）
2. 観測度数
3. 検定統計量の表（`pearson` の行を見るのが基本）

In [ ]:
expected, observed, chi2_stats = pg.chi2_independence(df, x="region", y="emp_type")
print("観測度数:")
print(observed)
print("\n期待度数:")
print(expected.round(1))
print("\n検定結果:")
print(chi2_stats.round(3))

`cramer`（Cramer's V）はカイ二乗検定の効果量で、0（無関係）〜 1（完全に関係）の値をとります（0.1: 小、0.3: 中、0.5: 大が目安）。
この例では地域と雇用形態は独立になるようにデータを作ったので、p 値は大きくなるはずです。
一方、教育年数と雇用形態には関係を入れてあります。

In [ ]:
_, observed2, chi2_stats2 = pg.chi2_independence(df, x="education", y="emp_type")
print(observed2)
print()
print(chi2_stats2.loc[chi2_stats2["test"] == "pearson"].round(3))

---
## 9. 回帰分析

### 9.1 線形回帰

`pg.linear_regression(X, y)` は、切片を自動で加えて最小二乗法で推定し、係数・標準誤差・p 値・決定係数・信頼区間を表にして返します。
説明変数が複数あるときは DataFrame を渡します。

In [ ]:
X = df[["education", "experience"]]
y = df["wage"]
print(pg.linear_regression(X, y).round(3))

カテゴリ変数（性別・地域・雇用形態）は **ダミー変数**（0/1 の列）に変換してから渡します。`pd.get_dummies()` を使い、
`drop_first=True` で基準カテゴリを 1 つ落とします（多重共線性を避けるため）。

In [ ]:
X_full = pd.get_dummies(
    df[["education", "experience", "gender", "region", "emp_type"]],
    drop_first=True, dtype=float,
)
print("説明変数:", list(X_full.columns))
reg = pg.linear_regression(X_full, df["wage"])
print(reg.round(3))
print("\n決定係数 R²:", round(reg["r2"].iloc[0], 3))

係数の読み方：`gender_男性` の係数は「他の条件が同じとき、男性は女性より年収が何万円高いか」を表します
（データを作るときに −45 万円の女性ペナルティを入れたので、45 前後になるはずです）。

### 9.2 ロジスティック回帰

結果が「正規雇用かどうか」のような **2 値** のときはロジスティック回帰を使います。
`pg.logistic_regression(X, y)` の `y` は 0/1 の整数で渡します。

In [ ]:
y_regular = (df["emp_type"] == "正規").astype(int)
X_logit = df[["education", "experience"]]
import warnings
with warnings.catch_warnings():
    # pingouin 内部の scikit-learn 呼び出しが出す無害な UserWarning（penalty=None と C の併用）を抑える
    warnings.simplefilter("ignore", UserWarning)
    logit = pg.logistic_regression(X_logit, y_regular)
print(logit.round(3))
print("\nオッズ比（係数の指数）:")
print(np.exp(logit.set_index("names")["coef"]).round(3))

オッズ比が 1 より大きければ「その変数が 1 増えると正規雇用になる確率（オッズ）が上がる」ことを意味します。
教育年数が正規雇用を増やすようにデータを作ったので、`education` のオッズ比は 1 を超えるはずです。

### 練習問題 5

1. 年収を、教育年数・勤続年数・勤続年数の 2 乗（`experience_sq` 列を作る）で回帰してください。2 乗項の係数の符号を確認しましょう。
2. 説明変数に性別と地域のダミー変数も加えて回帰し、決定係数 R² が 1 と比べてどれだけ上がったか確認してください。
3. 「年収が 500 万円以上かどうか」を 0/1 の変数にして、教育年数と勤続年数でロジスティック回帰を行い、オッズ比を求めてください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
X1 = df[["education", "experience"]].copy()
X1["experience_sq"] = X1["experience"] ** 2
r1 = pg.linear_regression(X1, df["wage"])
print(r1.round(3))

# 2
X2 = pd.get_dummies(df[["education", "experience", "gender", "region"]], drop_first=True, dtype=float)
X2["experience_sq"] = X2["experience"] ** 2
r2 = pg.linear_regression(X2, df["wage"])
print("R²:", round(r1["r2"].iloc[0], 3), "→", round(r2["r2"].iloc[0], 3))

# 3
y_high = (df["wage"] >= 500).astype(int)
lg = pg.logistic_regression(df[["education", "experience"]], y_high)
print(lg.round(3))
print(np.exp(lg.set_index("names")["coef"]).round(3))
```

</details>

---
## 10. ベイズファクター

p 値は「帰無仮説のもとでこのデータが得られる確率」であって、「帰無仮説が正しい確率」ではありません。
**ベイズファクター BF10** は「対立仮説 H1 と帰無仮説 H0 のどちらをデータがどれだけ支持しているか」の比で、
p 値より直感的に解釈できます。

| BF10 | 解釈 |
|---|---|
| 1 〜 3 | H1 をわずかに支持 |
| 3 〜 10 | H1 を中程度に支持 |
| 10 〜 30 | H1 を強く支持 |
| 30 以上 | H1 を非常に強く支持 |
| 1/3 〜 1 | H0 をわずかに支持 |
| 1/3 未満 | H0 を中程度以上に支持 |

`pg.ttest()` や `pg.corr()` の結果に含まれる `BF10` 列がこれです。t 統計量と標本サイズから直接計算することもできます。

In [ ]:
t_value = pg.ttest(male, female)["T"].iloc[0]
bf = pg.bayesfactor_ttest(t_value, nx=len(male), ny=len(female))
print("t =", round(t_value, 3), " BF10 =", round(bf, 2))

# 同じ分布から作った（差のない）2 群では BF10 は小さくなる（1 前後なら「どちらの証拠にもならない」、1 を下回れば帰無仮説寄り）
same_a = rng.normal(50, 10, 30)
same_b = rng.normal(50, 10, 30)
print(pg.ttest(same_a, same_b)[["T", "p_val", "BF10"]].round(3))

---
## 11. まとめ

| 目的 | 関数 | 主な列 |
|---|---|---|
| 正規性・等分散性 | `pg.normality()`, `pg.homoscedasticity()` | `pval`, `normal` / `equal_var` |
| 2 群の平均比較 | `pg.ttest()`（`paired=True` で対応あり） | `p_val`, `CI95`, `cohen_d`, `power`, `BF10` |
| 効果量・検出力 | `pg.compute_effsize()`, `pg.power_ttest()` | — |
| ノンパラメトリック | `pg.mwu()`, `pg.wilcoxon()`, `pg.kruskal()` | `p_val`, `RBC`, `CLES` |
| 分散分析 | `pg.anova()`, `pg.welch_anova()`, `pg.rm_anova()` | `F`, `p_unc`, `np2` |
| 多重比較 | `pg.pairwise_tests()`, `pg.pairwise_tukey()` | `p_corr` / `p_tukey`, `hedges` |
| 相関 | `pg.corr()`, `pg.pairwise_corr()`, `pg.partial_corr()` | `r`, `CI95`, `p_val` |
| カテゴリ変数の関係 | `pg.chi2_independence()` | `chi2`, `pval`, `cramer` |
| 回帰 | `pg.linear_regression()`, `pg.logistic_regression()` | `coef`, `se`, `pval`, `r2` |
| ベイズファクター | `pg.bayesfactor_ttest()` | — |

### 報告のしかた

検定結果を書くときは、**統計量・自由度・p 値・効果量・信頼区間** をセットで報告します。
例：「男性と女性の平均年収には有意な差があった（Welch の t 検定, t(…) = …, p < .001, d = …, 95% CI […, …]）。」

## 次のステップ

- `python/scipy/scipy_stats_intermediate_tutorial.ipynb` — 検定の理論的背景をより詳しく
- `python/statsmodels/statsmodels_tutorial.ipynb` — 回帰分析を本格的に
- `python/statsmodels/panel_data_beginner_tutorial.ipynb` — パネルデータ（固定効果・操作変数）

---
## 総合演習：職業訓練プログラムの効果検証

ある自治体が、失業者向けの職業訓練プログラムの効果を調べるため、参加者（処置群）60 人と
非参加者（対照群）60 人について、プログラム前後の月収（万円）を記録しました。次のセルで合成データを作ります。

1. 前後の月収の変化量（`gain = after - before`）を計算し、処置群と対照群それぞれで正規性を確認してください。
2. 処置群と対照群の変化量に差があるか、Welch の t 検定で検定し、Cohen's d と 95% 信頼区間を報告してください。
3. 2 で得た効果量を検出力 0.8 で検出するには、1 群あたり何人必要だったか計算してください。
4. `after` を `before` と処置ダミー（`treated`）で回帰し、事前の月収を調整したうえでのプログラム効果（`treated` の係数）を求めてください。
5. 処置群・対照群の前後の平均月収を折れ線グラフで比較してください。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください
rng_ex = np.random.default_rng(2024)
n_each = 60
before_t = rng_ex.normal(18, 4, n_each)
before_c = rng_ex.normal(18, 4, n_each)
program = pd.DataFrame({
    "treated": np.r_[np.ones(n_each, dtype=int), np.zeros(n_each, dtype=int)],
    "before": np.r_[before_t, before_c].round(1),
    "after": np.r_[before_t + 2.5 + rng_ex.normal(0, 3, n_each), before_c + 0.5 + rng_ex.normal(0, 3, n_each)].round(1),
})
print(program.head())

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
program["gain"] = program["after"] - program["before"]
gain_t = program.loc[program["treated"] == 1, "gain"]
gain_c = program.loc[program["treated"] == 0, "gain"]

# 1. 正規性
print(pg.normality(data=program, dv="gain", group="treated").round(3))

# 2. Welch の t 検定
res = pg.ttest(gain_t, gain_c)
print(res.round(3))
print(f"\n効果量 d = {res['cohen_d'].iloc[0]:.3f}, 95% CI = {res['CI95'].iloc[0]}")

# 3. 必要な標本サイズ
needed = pg.power_ttest(d=res["cohen_d"].iloc[0], power=0.8, alpha=0.05)
print("検出力 0.8 に必要な 1 群あたりの人数:", round(needed))

# 4. 事前の月収を調整した回帰
reg = pg.linear_regression(program[["before", "treated"]], program["after"])
print(reg.round(3))
print("プログラム効果（treated の係数）:", round(reg.loc[reg["names"] == "treated", "coef"].iloc[0], 2), "万円")

# 5. 折れ線グラフ
means = program.groupby("treated")[["before", "after"]].mean()
plt.figure(figsize=(6, 4))
for treated, label in [(1, "処置群（参加）"), (0, "対照群（非参加）")]:
    plt.plot(["プログラム前", "プログラム後"], means.loc[treated], marker="o", label=label)
plt.ylabel("平均月収（万円）")
plt.title("職業訓練プログラムの前後比較")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

お疲れさまでした！ pingouin を使うと、検定・効果量・信頼区間・検出力をまとめて得られるので、
「有意かどうか」だけでなく「どれくらいの差なのか」まで踏み込んだ報告ができます。